# 08 — Automated Commentary & Report

Turns the metrics into research-note prose, e.g. *"Berkshire increased exposure to Chevron while trimming Occidental. Apple remained the largest holding, representing 24.8% of assets. Portfolio turnover declined compared with the previous quarter."*

Every sentence is generated from the data — swap the CIK in `config.py` and the same code narrates any manager. The full markdown report is written to `reports/`.

In [ ]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

In [ ]:
from src import commentary, config
from src.utils import load_parquet

holdings = load_parquet(config.HOLDINGS_PARQUET)
tx = load_parquet(config.TRANSACTIONS_PARQUET)
summary = load_parquet(config.PORTFOLIO_PARQUET)

latest = summary["quarter"].iloc[-1]
print(commentary.quarterly_commentary(latest, holdings, tx, summary))

In [ ]:
report_md = commentary.full_report(holdings, tx, summary)
out = config.PROJECT_ROOT / "reports" / "portfolio_review.md"
out.write_text(report_md, encoding="utf-8")
print("wrote", out)
print()
print(report_md[:1500])

### Optional next step — Parquet → Supabase
The `data/processed/*.parquet` files are the project's "database". To productionize, replace the parquet layer with Supabase/BigQuery tables: each notebook writes to a table instead of a file, everything else stays identical — that's the payoff of the *notebooks depend on data + functions, never on each other* rule.